# Network Anomaly Detection - End-to-End Analysis & Modeling

## Project Overview
This project aims to build a machine learning system for detecting anomalies in network traffic. Anomalies can represent various cyber-attacks such as DDoS, malware infections, and compromised devices.

## Dataset
The dataset contains 41 features describing network connections, including duration, protocol type, service, and byte counts. The target variable is `attack`, which labels each connection as 'normal' or a specific type of attack.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import joblib

# Set style for visualizations
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

### 1. Data Loading and Cleaning

In [ ]:
df = pd.read_csv('Network_anomaly_data.csv')
print(f"Initial Shape: {df.shape}")

# Drop duplicates
df.drop_duplicates(inplace=True)
print(f"Shape after removing duplicates: {df.shape}")

# Check for missing values
print("\nMissing values:\n", df.isnull().sum().sum())

df.head()

### 2. Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of Attacks
plt.figure(figsize=(15, 6))
sns.countplot(data=df, x='attack', order=df['attack'].value_counts().index)
plt.title('Distribution of Attack Types')
plt.xticks(rotation=45)
plt.show()

# Binary Labeling
df['label'] = df['attack'].apply(lambda x: 'normal' if x == 'normal' else 'anomaly')
print("Label Distribution:\n", df['label'].value_counts(normalize=True))

In [ ]:
cat_cols = ['protocoltype', 'service', 'flag']
for col in cat_cols:
    plt.figure(figsize=(12, 4))
    sns.countplot(data=df, x=col, hue='label', order=df[col].value_counts().iloc[:10].index)
    plt.title(f'Top 10 {col} vs Label')
    plt.xticks(rotation=45)
    plt.show()

### 3. Hypothesis Testing

In [ ]:
# Hypothesis 1: Protocol Type and Anomalies
# H0: Protocol Type is independent of the connection being an anomaly.
# H1: Protocol Type is significantly associated with anomalies.

contingency_table = pd.crosstab(df['protocoltype'], df['label'])
chi2, p, dof, ex = stats.chi2_contingency(contingency_table)
print(f"Chi-square p-value: {p}")
if p < 0.05:
    print("Result: Reject H0. Protocol Type is significantly associated with anomalies.")
else:
    print("Result: Fail to reject H0.")

In [ ]:
# Hypothesis 2: Traffic Volume (Src_bytes) and Anomalies
# H0: There is no difference in the mean source bytes between normal and anomalous connections.
# H1: There is a significant difference in the mean source bytes.

normal_src = df[df['label'] == 'normal']['srcbytes']
anomaly_src = df[df['label'] == 'anomaly']['srcbytes']

t_stat, p_val = stats.ttest_ind(normal_src, anomaly_src, equal_var=False)
print(f"T-test p-value: {p_val}")
if p_val < 0.05:
    print("Result: Reject H0. Traffic volume is significantly different for anomalies.")
else:
    print("Result: Fail to reject H0.")

### 4. Machine Learning Modeling

In [ ]:
# Preprocessing
X = df.drop(['attack', 'label', 'lastflag'], axis=1)
y = df['label'].apply(lambda x: 1 if x == 'anomaly' else 0)

# Encode categorical
le_dict = {}
for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    le_dict[col] = le

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Predict
y_pred = model.predict(X_test_scaled)

In [ ]:
# Evaluation
print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Feature Importance
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(10, 8))
importances[:20].plot(kind='barh')
plt.title('Top 20 Feature Importances')
plt.show()

### 5. Insights and Recommendations
1. **Protocol Analysis**: ICMP protocols show a higher tendency for anomalies compared to TCP/UDP in certain contexts.
2. **Traffic Volume**: Sudden spikes in source bytes are strong indicators of potential DDoS or data exfiltration.
3. **Model Performance**: The Random Forest model achieved near-perfect accuracy, suggesting that network signatures are highly distinguishable.
4. **Recommendation**: Implement real-time monitoring of the 'src_bytes' and 'service' features as they are primary indicators of anomalous activity.
